In [ ]:
"""
This imports Python’s math module
We use it for math.inf  --->  math.inf means positive infinity , -math.inf means negative infinity
These are useful in Alpha-Beta pruning when we want to start with the biggest or smallest possible values
"""
import math

"""
This function checks how good the current board position is
It returns a number called a score
A higher score means the board is better for White (AI)
A lower score means the board is worse for White
"""

def evaluate_board(board):
    """Evaluate the current board position and return a score"""
    # Assign positive scores for winning positions and negative for losing
    # If Black Is CheckMated Then White Wins (Good For White)
    if is_checkmate(board, 'Black'):
        return 100  # AI wins
    # If White Is CheckMated Then White Has Lost (Good For Black)
    elif is_checkmate(board, 'White'):
        return -100  # Human wins

    # Heuristic evaluation: favor positions where the opponent's king has fewer moves
    # Check The Possible Moves Can Black Take (Gives A List Of Moves)
    # Fewer Black moves usually means White is controlling the game better
    black_king_moves = len(generate_legal_moves(board, 'Black'))
    # Get Row And Column Of BlackKing
    black_row,black_col=board['BlackKing']
    # Check How Close BlackKing To The Edge i-e Smaller Value means Closer To Edge Which Is Good For CheckMate
    # In king and queen vs king, pushing the enemy king to the edge is good
    edge_score=min(black_row,7-black_row)+min(black_col,7-black_col)
    # Less Moves Means High Score , Closer To Edge Means High Score
    # Final result: White gets a better score when Black has fewer moves and is trapped near the edge
    # black_moves * 5 means if Black has many moves, we reduce White’s score
    # edge_score is subtracted too, so positions where Black is closer to the edge become better
    # edge_score is subtracted too, so positions where Black is closer to the edge become better
    return 50 - black_king_moves*5-edge_score  # Favor positions with fewer escape options for Black

# This function checks whether the White queen can attack a target square
# It follows queen movement rules in chess
def queen_attacks(queen_pos,target_pos,board):
  # Exact Row/Column For Queen
  queen_row,queen_col=queen_pos
  # # Exact Row/Column For Target Square’s
  target_row,target_col=target_pos

  # This means the queen is not on the board
  # We used (-1, -1) earlier to show queen was captured
  # If queen is gone, it cannot attack anything
  if queen_pos==(-1,-1):
    return False
  # If Queen And Target Are In Same Row If yes, queen might attack horizontally
  if queen_row==target_row:
    # Then Decide Direction Left Or Right If target is to the right, step is 1
    # If target is to the left, step is -1
    if target_col>queen_col:
      step=1
    else:
      step=-1

    # Check All The Squares Between Queen And Target because queen cannot jump over pieces
    for i in range(queen_col+step,target_col,step):
      # If a king is blocking the path, queen cannot attack through it.So we return False
      if (queen_row,i)==board['WhiteKing'] or (queen_row,i)==board['BlackKing']:
        return False

    # If no piece blocks the row path, queen attacks the target, So we return True
    return True

  # This checks if queen and target are in the same column If yes, queen might attack vertically
  if queen_col==target_col:
    # This decides whether queen moves up or down
    if target_row>queen_row:
      step=1
    else:
      step=-1

    # Check The Squares Between Queen And Target Vertically
    for j in range(queen_row+step,target_row,step):
      # If Piece Blocks The Path Then Queen Attack Will Not Possible
      if (j,queen_col)==board['WhiteKing'] or (j,queen_col)==board['BlackKing']:
        return False

    # No Blocking Piece, So Queen Can Attack
    return True

  # This checks if target is on a diagonal from the queen
  # For diagonal movement, row difference and column difference must be equal
  if abs(queen_row-target_row)==abs(queen_col-target_col):
    # Finding The Direction In Row
    if target_row>queen_row:
      row_step=1
    else:
      row_step=-1

    # Finding The Direction In Col
    if target_col>queen_col:
      col_step=1
    else:
      col_step=-1

    # Start checking from the square next to the queen In Row And Col Both
    row=queen_row+row_step
    col=queen_col+col_step
    # Step By Step Movement Keep checking squares until we reach the target
    while row!=target_row and col!=target_col:
      # If a piece blocks the diagonal, queen cannot attack
      if(row,col)==board['WhiteKing'] or (row,col)==board['BlackKing']:
        return False # Stop If Blocked
      # Move one step more along the diagonal
      row+=row_step
      col+=col_step
    # If no piece blocked the path, queen attacks the target
    return True

  # If target is not in same row, column, or diagonal, queen cannot attack it
  return False

# This checks whether a square is attacked by White’s pieces
def is_square_attacked(board,squre):
  """ Check If A Square Is Attacked By White """
  # This gets White king and White queen positions
  WhiteKing=board['WhiteKing']
  WhiteQueen=board['WhiteQueen']

  # A king attacks all 8 surrounding squares
  # This line checks whether the square is one step away from White king
  # If yes, that square is attacked
  if max(abs(WhiteKing[0]-squre[0]),abs(WhiteKing[1]-squre[1]))==1:
    return True

  # This checks if White queen attacks that square If yes, square is attacked
  if queen_attacks(WhiteQueen,squre,board):
    return True

  # If neither White king nor White queen attacks the square, return False
  return False


# This function creates all valid moves for the given player. If player is White,
# it generates White king and White queen moves.
# If player is Black, it generates Black king moves.

def generate_legal_moves(board, player):
    moves_list=[]  # make empty list to store all possible moves

    if player=='White':  # check if its white's turn
      WhiteQueen=board['WhiteQueen']  # get queen position
      WhiteKing=board['WhiteKing']  # get white king position
      BlackKing=board['BlackKing']  # get black king position

      # all 8 directions a king can move on the board
      King_Directions=[(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]

      # try moving the white king in each direction
      for i,j in King_Directions:
        new_row=WhiteKing[0]+i  # add direction to current row
        new_col=WhiteKing[1]+j  # add direction to current column

        if 0<=new_row<=7 and 0<=new_col<=7:  # make sure its on the board
          if (new_row,new_col)!=WhiteKing and (new_row,new_col)!=BlackKing:  # make sure square is not occupied
            if max(abs(new_row-BlackKing[0]),abs(new_col-BlackKing[1]))>1:  # king cant be right next to black king
              moves_list.append(('WhiteKing',(new_row,new_col)))  # add this move if its legal

      # directions the queen can move - horizontal, vertical, and diagonals
      Queen_Directions=[(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]

      for i,j in Queen_Directions:  # for each direction
        step=1  # start 1 square away
        while True:  # keep going until we hit something
          new_row=WhiteQueen[0]+i*step  # calculate row
          new_col=WhiteQueen[1]+j*step  # calculate column

          if not(0<=new_row<=7 and 0<=new_col<=7):  # if off the board, stop
            break
          if (new_row,new_col)==WhiteKing or (new_row,new_col)==BlackKing:  # if hit a king, stop
            break
          if max(abs(new_row-BlackKing[0]),abs(new_col-BlackKing[1]))>1:  # queen cant let white king be next to black king
            moves_list.append(('WhiteQueen',(new_row,new_col)))  # add this square as possible move
          step+=1  # go further in this direction

    else:  # its black's turn
      BlackKing=board['BlackKing']  # get black king position
      WhiteKing=board['WhiteKing']  # get white king position
      WhiteQueen=board['WhiteQueen']  # get white queen position
      King_Directions=[(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]  # 8 directions

      for i,j in King_Directions:  # try each direction
        new_row=BlackKing[0]+i  # new position
        new_col=BlackKing[1]+j

        if 0<=new_row<=7 and 0<=new_col<=7:  # on the board?
          if max(abs(new_row-WhiteKing[0]),abs(new_col-WhiteKing[1]))>1:  # not next to white king?
            temp_board=apply_move(board,('BlackKing',(new_row,new_col)))  # imagine this move happened
            if not is_square_attacked(temp_board,(new_row,new_col)):  # is black king safe there?
              moves_list.append(('BlackKing',(new_row,new_col)))  # yes so add it

    return moves_list


def apply_move(board, move):
    # unpack the move tuple so we know which piece and where it goes
    piece, new_position = move

    # copy the board so we dont change the original
    new_board = board.copy()

    # update the board with the new position
    if piece == 'WhiteKing':
      new_board['WhiteKing'] = new_position
    elif piece == 'BlackKing':
      new_board['BlackKing'] = new_position
    elif piece == 'WhiteQueen':
      new_board['WhiteQueen'] = new_position

    return new_board

def is_checkmate(board, player):
    # see if there are any legal moves left for the player
    if player=='White':
      # if white has no moves, its checkmate
      if not generate_legal_moves(board,'White'):
        return True

    if player=='Black':
      # if black has no moves, its checkmate
      if not generate_legal_moves(board,'Black'):
        return True

    return False

def alpha_beta_pruning(board, depth, alpha, beta, current_player):
    # if we reached the bottom of search tree or game is over, return the score
    if depth == 0 or is_checkmate(board,'White') or is_checkmate(board,'Black'):
        return evaluate_board(board)

    # white is trying to get the highest score
    if current_player=='White':
        max_eval=-math.inf  # start with worst possible score
        for move in generate_legal_moves(board,'White'):  # for each possible move
            new_board=apply_move(board, move)  # pretend we made this move
            eval=alpha_beta_pruning(new_board, depth-1, alpha, beta,'Black')  # see how good it is
            max_eval=max(max_eval, eval)  # remember best score so far
            alpha=max(alpha, eval)  # update alpha cutoff
            if beta<=alpha:  # if black wouldnt let us get here anyway, skip
                break
        return max_eval

    # black is trying to get the lowest score (bad for white)
    else:
      min_eval=math.inf  # start with worst possible for black
      for move in generate_legal_moves(board,'Black'):  # for each possible move
        new_board=apply_move(board, move)  # pretend we made this move
        eval=alpha_beta_pruning(new_board, depth-1, alpha, beta,'White')  # see how good it is for black
        min_eval=min(min_eval, eval)  # remember worst score for white
        beta=min(beta, eval)  # update beta cutoff
        if beta<=alpha:  # if white wouldnt let us get here anyway, stop
          break
      return min_eval

def find_best_move(board, player='White'):
    # variables to track the best move we find
    best_value=-math.inf  # start with worst score
    best_move=None  # no move chosen yet
    opponent = 'Black' if player == 'White' else 'White'  # who are we playing against

    # white wants to get the highest score
    if player == 'White':
      for move in generate_legal_moves(board, player):  # try each move
        new_board = apply_move(board, move)  # see what board looks like after this move
        move_value = alpha_beta_pruning(new_board, 3, -math.inf, math.inf, opponent)  # how good is it? (reduced to depth 3 for speed)
        if move_value > best_value:  # is this the best move so far?
          best_value = move_value
          best_move = move

    # black wants to get the lowest score (bad for white)
    else:
      best_value=math.inf  # start with highest score (worst for black)
      for move in generate_legal_moves(board, player):  # try each move
        new_board=apply_move(board, move)  # see what board looks like after this move
        move_value=alpha_beta_pruning(new_board, 3, -math.inf, math.inf, opponent)  # how good is it for black? (reduced to depth 3 for speed)
        if move_value < best_value:  # is this better? (lower is better for black)
          best_value=move_value
          best_move=move

    return best_move  # return what we think is the best move

def play_chess_endgame(white_king_pos=(3, 4), white_queen_pos=(4, 5), black_king_pos=(7, 7)):
    # set up the starting position with custom coordinates
    board = {'WhiteKing': white_king_pos, 'WhiteQueen': white_queen_pos, 'BlackKing': black_king_pos}
    move_count = 0  # count how many moves have been made
    max_moves = 50  # reduced from 100 for faster testing
    move_history = []  # track moves to detect cycles

    print(f"\n=== Game Start: WhiteKing={white_king_pos}, WhiteQueen={white_queen_pos}, BlackKing={black_king_pos} ===")

    while move_count < max_moves:  # keep playing until we hit the limit
      # white makes a move
      best_move = find_best_move(board, 'White')  # find whites best move

      if best_move is None:  # if white has no moves
        print("White has no moves - Stalemate!")
        break

      # Check for draw by detecting repeated move sequences (4+ moves repeating)
      if len(move_history) >= 8:  # need at least 8 moves to check for 4-move cycle
        # Check if the last 4 moves match the 4 moves before that (4-move cycle repeating)
        if move_history[-4:] == move_history[-8:-4]:
          print("Game is in a draw - 4-move cycle detected")
          break

      print("White's move:", best_move)  # show what white is doing
      board = apply_move(board, best_move)  # actually make the move
      move_history.append(best_move)  # record the move
      move_count += 1  # one more move done

      if is_checkmate(board, 'Black'):  # did white win?
        print("White Wins!")
        break

      # black makes a move
      best_move = find_best_move(board, 'Black')  # find blacks best move

      if best_move is None:  # if black has no moves
        print("Black has no moves - Black is Checkmated or Stalemated!")
        break

      print("Black's move:", best_move)  # show what black is doing
      board = apply_move(board, best_move)  # actually make the move
      move_history.append(best_move)  # record the move
      move_count += 1  # one more move done

      if is_checkmate(board, 'White'):  # did black somehow win?
        print("Black Wins!")
        break

    if move_count >= max_moves:  # did we stop because of the limit?
      print(f"Game stopped after {max_moves} moves (move limit reached)")

    print(f"Total moves: {move_count}\n")

if __name__ == "__main__":
    print("=" * 70)

    # Test 1: Quick White Win
    print("[TEST 1] Quick White Win - Black King Trapped in Corner")
    play_chess_endgame((3, 4), (4, 5), (7, 7))

    # Test 2: Moderate Win
    print("[TEST 2] Moderate White Win - Black King Near Corner")
    play_chess_endgame((2, 2), (3, 3), (7, 6))

    # Test 3: Longer Battle
    print("[TEST 3] Extended Game - White Wins After Multiple Moves")
    play_chess_endgame((1, 1), (2, 3), (4, 4))

    # Test 4: Very Quick Win
    print("[TEST 4] Instant Checkmate - Optimal Queen Position")
    play_chess_endgame((3, 3), (4, 4), (0, 7))

    # Test 5: Strategic Chase
    print("[TEST 5] Strategic Pursuit - Black King Far Away")
    play_chess_endgame((0, 0), (1, 1), (6, 6))

    # Test 6: Difficult Position for White
    print("[TEST 6] Challenging Position - Pieces Far Apart")
    play_chess_endgame((0, 2), (1, 4), (7, 5))

    # Test 7: Draw Scenario - Likely Cycle
    print("[TEST 7] Draw Scenario - Repetitive Positions (3-fold Repetition)")
    play_chess_endgame((4, 4), (5, 4), (7, 7))

    # Test 8: Corner Trap with Draw
    print("[TEST 8] Corner Trap - Draw by Cycle")
    play_chess_endgame((5, 6), (6, 7), (7, 5))

    # Test 9: Central Position
    print("[TEST 9] Central Battle - Draw by Long Cycle")
    play_chess_endgame((3, 3), (3, 4), (5, 5))

    # Test 10: Edge Case - Kings Adjacent
    print("[TEST 10] Kings Near Each Other - Quick White Win")
    play_chess_endgame((5, 5), (5, 7), (6, 7))

    # Test 11: THEORETICAL - King vs King (No Queen)
    print("[TEST 11] THEORETICAL CASE - King vs King Only (No Queen)")
    print("         This shows what happens when White has no Queen")
    play_chess_endgame((3, 3), (-1, -1), (7, 7))  # Queen at (-1,-1) = missing/captured

    # Test 12: Black in Fortified Position
    print("[TEST 12] Fortified Defense - Black Surrounded by Own King Mobility")
    play_chess_endgame((7, 0), (7, 2), (0, 0))

    # Test 13: Queen Sacrifice Position (White Queen Far Away)
    print("[TEST 13] Queen Distant - Long Chase Game")
    play_chess_endgame((4, 0), (0, 0), (7, 7))

    # Test 14: Minimal Mating Net
    print("[TEST 14] Minimal Mating Net - Very Restricted Space")
    play_chess_endgame((6, 6), (7, 7), (0, 0))

    # Test 15: Black's Best Defense Position
    print("[TEST 15] Black's Best Defense - Central King Position")
    play_chess_endgame((1, 1), (3, 3), (4, 4))

    print("\n" + "=" * 70)


Streaming output truncated to the last 5000 lines.
Checking move for WhiteKing to (1, 3)
Checking move for WhiteKing to (1, 4)
Checking move for WhiteKing to (1, 5)
Generating moves for WhiteKing at: (0, 4)
Checking move for WhiteKing to (-1, 3)
Checking move for WhiteKing to (-1, 4)
Checking move for WhiteKing to (-1, 5)
Checking move for WhiteKing to (0, 3)
Checking move for WhiteKing to (0, 5)
Checking move for WhiteKing to (1, 3)
Checking move for WhiteKing to (1, 4)
Checking move for WhiteKing to (1, 5)
Generating moves for WhiteKing at: (0, 4)
Checking move for WhiteKing to (-1, 3)
Checking move for WhiteKing to (-1, 4)
Checking move for WhiteKing to (-1, 5)
Checking move for WhiteKing to (0, 3)
Checking move for WhiteKing to (0, 5)
Checking move for WhiteKing to (1, 3)
Checking move for WhiteKing to (1, 4)
Checking move for WhiteKing to (1, 5)
Generating moves for WhiteKing at: (0, 4)
Checking move for WhiteKing to (-1, 3)
Checking move for WhiteKing to (-1, 4)
Checking move for

KeyboardInterrupt: 